# **Fase 2: Feature Engineering**

Estudiante: Maria Camila Navarrete Pinzón

Código: 2294353

Fecha: 05 febrero, 2026

# Notebook 3: Pipelines y Feature Engineering

Este cuaderno tiene como objetivo aplicar VectorAssembler y construir un pipeline de transformación.

**Conceptos clave:** 

- Transformer: Aplica transformaciones
- Estimator: Aprende de los datos y genera un modelo
- Pipeline: Encadena múltiples stages secuencialmente

**Actividades:**
1. Crear StringIndexer para variables categóricas
2. Aplicar OneHotEncoder
3. Combinar features con VectorAssembler
4. Construir y ejecutar pipelina


## 1. Configuración de SparkSession

Se crea una sesión de spark configurada para ejecutarse en modo local, asignando memoria al driver.

In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
)
from pyspark.ml import Pipeline
from pyspark.sql.functions import col, when, isnull

spark = SparkSession.builder \
    .appName("SECOP_FeatureEngineering") \
    .master("local[*]") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/12 23:30:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 2. Carga de datos

In [2]:

df = spark.read.parquet("/opt/spark-data/processed/secop_eda.parquet")
print(f"Registros cargados: {df.count():,}")

print("Columnas disponibles:")
for col_name in df.columns:
    print(f"  - {col_name}")

Registros cargados: 52,248
Columnas disponibles:
  - referencia_del_contrato
  - nit_entidad
  - nombre_entidad
  - departamento
  - ciudad
  - tipo_de_contrato
  - valor_del_contrato
  - fecha_de_firma
  - estado_contrato
  - valor_del_contrato_num
  - fecha_de_firma_parsed
  - anio
  - mes


26/02/12 23:30:21 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


## 3. Selección de Features

**Objetivo**: Identificar las mejores variables para predecir el valor del contrato.

**Instrucciones**:
1. Analiza las columnas disponibles en el dataset
2. Selecciona al menos 3 variables categóricas relevantes
3. Selecciona al menos 2 variables numéricas relevantes
4. Justifica tu selección con un comentario

In [3]:

categorical_cols = [
    "departamento",         
    "tipo_de_contrato",      
    "estado_contrato"        
]

numeric_cols = [
    "plazo_de_ejec_del_contrato",  
    "valor_del_contrato_num"       
]

available_cat = [c for c in categorical_cols if c in df.columns]
available_num = [c for c in numeric_cols if c in df.columns]

print("Features seleccionados")
print(f"Variables categóricas disponibles: {available_cat}")
print(f"Variables numéricas disponibles: {available_num}")


Features seleccionados
Variables categóricas disponibles: ['departamento', 'tipo_de_contrato', 'estado_contrato']
Variables numéricas disponibles: ['valor_del_contrato_num']


**Variables categóricas**: 

- Departamento: Permite capturar diferencias regionales en contratación pública.
- Tipo de contrato: Influye directamente en el valor, duración y riesgos del contrato.
- Estado del contrato: Aporta informaciòn clave sobre el ciclo de vida y resultados del contrato.

**Variables numéricas**:

- Plazo de ejecución del contrato: Refleja la complejidad temporal del contrato.
- Valor del contrato: Es una de las variables mas relevantes y explicativas del análisis, además de ser una posible variable objetivo. 

Estas variables combinan contextos, características operativas y magnitud regional contractual, lo que las hace adecuadas a modelos de machine learning. 



## 4. Implementación de estrategias de limpieza de datos

**Pregunta**: ¿Qué estrategia usarás para manejar valores nulos?
- Opción A: Eliminar filas con nulos (dropna)
- Opción B: Imputar valores (usar Imputer)
- Opción C: Crear una categoría "DESCONOCIDO" para categóricas

**Opción A: Eliminar filas con valores nulos**

Se opta por eliminar las filas con valores nulos únicamente en las variables seleccionadas para el modelo, esto porque estas columnas son críticas para el ejercicio. Dado que el dataset cuenta con más de 52.000 registros, la pérdida de observaciones no compromete la representatividad del análisis y permite garantizar calidad en los datos, evitando sesgos introducidos por imputaciones.



In [4]:

df_clean = df.dropna(subset=available_cat + available_num)
print(f"Registros después de limpiar nulos: {df_clean.count():,}")

Registros después de limpiar nulos: 52,248


In [5]:
from pyspark.ml.feature import StringIndexer
indexers = [
    StringIndexer(
        inputCol=col,
        outputCol=col + "_idx",
        handleInvalid="keep"
    )
    for col in available_cat
]

print("StringIndexers creados:")
for idx in indexers:
    print(f"  - {idx.getInputCol()} -> {idx.getOutputCol()}")


StringIndexers creados:
  - departamento -> departamento_idx
  - tipo_de_contrato -> tipo_de_contrato_idx
  - estado_contrato -> estado_contrato_idx


### 4.1. OneHotEncoder

 **Concepto**: OneHotEncoder convierte índices en vectores binarios.


In [6]:
from pyspark.ml.feature import OneHotEncoder
encoders = [
    OneHotEncoder(
        inputCol=col + "_idx",
        outputCol=col + "_vec"
    )
    for col in available_cat
]
print("\nOneHotEncoders creados:")
for enc in encoders:
    print(f"  - {enc.getInputCol()} -> {enc.getOutputCol()}")



OneHotEncoders creados:
  - departamento_idx -> departamento_vec
  - tipo_de_contrato_idx -> tipo_de_contrato_vec
  - estado_contrato_idx -> estado_contrato_vec


## 5. Crear VectorAssembler para combinar features

inputCols debe incluir:
1. Variables numéricas originales (available_num)
2. Variables categóricas codificadas (con sufijo "_vec")

In [7]:
from pyspark.ml.feature import VectorAssembler

feature_cols = (
    available_num +
    [col + "_vec" for col in available_cat]
)

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_raw"
)

print(f"\nVectorAssembler combinará {len(feature_cols)} features:")
for col in feature_cols:
    print(f"  - {col}")



VectorAssembler combinará 4 features:
  - valor_del_contrato_num
  - departamento_vec
  - tipo_de_contrato_vec
  - estado_contrato_vec


Se combinan variables numéricas originales y variables categóricas previamente codificadas con OneHotEncoder, ya que los modelos de ML en Spark requieren un único vector numérico de entrada.

## 6. Construir Pipeline completo

**Objetivo**: Encadenar todos los transformadores en un Pipeline.

**Pregunta**: ¿Cuál es el orden correcto de los stages?


In [8]:
from pyspark.ml import Pipeline
pipeline_stages = (
    indexers +
    encoders +
    [assembler]
)

pipeline = Pipeline(stages=pipeline_stages)

print(f"\nPipeline con {len(pipeline_stages)} stages:")
for i, stage in enumerate(pipeline_stages):
    print(f"  Stage {i+1}: {type(stage).__name__}")



Pipeline con 7 stages:
  Stage 1: StringIndexer
  Stage 2: StringIndexer
  Stage 3: StringIndexer
  Stage 4: OneHotEncoder
  Stage 5: OneHotEncoder
  Stage 6: OneHotEncoder
  Stage 7: VectorAssembler


El orden correcto es **StringIndexer - OneHotEcoder - VectorAssembler**

Esto porque el OneHotEncoder depende de los ìndices creados por StringIndexer, el VectorAssembler depende de que todas las columnas ya sean numèricas y el pipeline debe estar sujeto a estas condiciones para no fallar. 

In [9]:
pipeline


Pipeline_38f98066d72c

## 7. Reto Bonus 1: Calcular dimension total de features post-encoding


In [10]:

print("\nEntrenando pipeline...")
pipeline_model = pipeline.fit(df_clean)
print("Pipeline entrenado exitosamente")


Entrenando pipeline...


Pipeline entrenado exitosamente


In [11]:
df_transformed = pipeline_model.transform(df_clean)
print("\nTransformación completada")
print(f"Columnas después de transformar: {len(df_transformed.columns)}")


Transformación completada
Columnas después de transformar: 20


In [12]:
df_transformed.select("features_raw").printSchema()

root
 |-- features_raw: vector (nullable = true)



Con esta línea de código (=true) confirmamos que es un vestor de spark ML y esta listo para el modelado.

In [13]:
sample_features = df_transformed.select("features_raw").first()[0]
print(f"Dimensión del vector de features: {len(sample_features)}")
df_transformed.select(
    available_cat[0] if available_cat else "id",
    available_cat[0] + "_idx" if available_cat else "id",
    available_cat[0] + "_vec" if available_cat else "id",
    "features_raw"
).show(5, truncate=True)

Dimensión del vector de features: 63
+--------------------+----------------+----------------+--------------------+
|        departamento|departamento_idx|departamento_vec|        features_raw|
+--------------------+----------------+----------------+--------------------+
|                Meta|             9.0|  (34,[9],[1.0])|(63,[0,10,35,56],...|
|              Boyacá|             6.0|  (34,[6],[1.0])|(63,[0,7,37,56],[...|
|           Santander|             5.0|  (34,[5],[1.0])|(63,[0,6,35,56],[...|
|Distrito Capital ...|             0.0|  (34,[0],[1.0])|(63,[0,1,35,56],[...|
|           Magdalena|             3.0|  (34,[3],[1.0])|(63,[0,4,35,56],[...|
+--------------------+----------------+----------------+--------------------+
only showing top 5 rows



La verificación del pipeline confirma que tras aplicar los procesos de indexación, codificación y ensamblaje, la dimensión final del vector es de 63. Esto indica que el conjunto de variables originales, junto con las columnas categóricas (por ejemplo, la variable departamento, que genera un vector disperso de dimensión 34), se integran correctamente en un único vector numérico. La estructura dispersa observada en los ejemplos refleja una representación eficiente de las variables categóricas, donde solo se activan las posiciones correspondientes a cada categoría.

## 8. Reto Bonus 2: Feature Importance Manual

**Objetivo**: Analizar la distribución de valores en el vector de features
**Instrucciones**:
1. Toma una muestra de 1000 registros
2. Convierte el vector de features a una matriz de Pandas
3. Calcula la varianza de cada feature
4. Identifica las top 5 features con mayor varianzaianza


In [14]:
import pandas as pd
import numpy as np

sample_df = (
    df_transformed
    .select("features_raw")
    .sample(fraction=0.01, seed=42) 
    .limit(1000)
    .toPandas()
)

print(f"Registros usados para el análisis: {len(sample_df)}")

features_matrix = np.array([
    row["features_raw"].toArray() for _, row in sample_df.iterrows()
])

print(f"Forma de la matriz de features: {features_matrix.shape}")
variances = np.var(features_matrix, axis=0)
top_5_idx = np.argsort(variances)[-5:][::-1]

print("\nTop 5 features con mayor varianza:")
for idx in top_5_idx:
    print(f"  Feature {idx}: varianza = {variances[idx]:.4f}")


Registros usados para el análisis: 576
Forma de la matriz de features: (576, 63)

Top 5 features con mayor varianza:
  Feature 0: varianza = 1091409000779401856.0000
  Feature 35: varianza = 0.2277
  Feature 56: varianza = 0.1934
  Feature 1: varianza = 0.1918
  Feature 57: varianza = 0.1233


El análisis realizado sobre 576 registros y 63 features, muestra que la feature 0 concentra una varianza extremadamente alta, lo que indica que es una variable con valores muy grandes que puede dominar el modelo si no se normaliza. En cambio, las features 35, 56, 1 y 57 presentan varianzas moderadas y similares, lo que sugiere que aportan información relevante y más equilibrada. En general el resultado señala la importancia de revisar el escalado de la feature 0 para evitar sesgos.

### Preguntas de reflexión

**Pipeline**
*¿Por qué usamos Pipeline en lugar de aplicar transformaciones individuales?*

Se usó pipeline porque permite encadenar todas las transformaciones en un solo flujo que se puede reproducir, asegurando que los mismos pasos se apliquen de forma consistente tanto en entrenamiento como en predicción, además de simplificar el código y reducir errores.

**Orden de transformaciones**
*¿Qué pasaría si aplicamos OneHotEncoder antes de StringIndexer?*

Si se aplica primero OneHotEncoder antes de StringIndexer, el proceso falla porque OneHotEncoder solo trabaja con índices numéricos. StringIndexer debe convertir primero las categorías en valores numéricos.

**StandardScaler**
*¿Cuándo usarías StandardScaler en el pipeline?*

Cuando las variables numéricas tienen escalas muy diferentes o cuando el modelo es sensible a la magnitud de los valores, como regresión logística.

**Guardar pipeline**
*¿Qué ventaja tiene guardar el pipeline_model en lugar del DataFrame transformado?*

Guardar el pipeline permite reutilizar exactamente las mismas transformaciones en nuevos datos, garantizando consistencia, eficiencia y evitando reprocesar todo el feature desde cero.



In [15]:
print("RESUMEN FEATURE ENGINEERING")
print(f"✓ Variables categóricas procesadas: {len(available_cat)}")
print(f"✓ Variables numéricas: {len(available_num)}")
print(f"✓ Dimensión final del vector: {len(sample_features)}")
print(f"✓ Pipeline guardado y listo para usar")

RESUMEN FEATURE ENGINEERING
✓ Variables categóricas procesadas: 3
✓ Variables numéricas: 1
✓ Dimensión final del vector: 63
✓ Pipeline guardado y listo para usar


#### Dataset guardado

In [17]:
import shutil
import os

pipeline_path = "/opt/spark-data/processed/feature_pipeline"

if os.path.exists(pipeline_path):
    shutil.rmtree(pipeline_path)

pipeline_model.save(pipeline_path)
print(f"\nPipeline guardado en: {pipeline_path}")



Pipeline guardado en: /opt/spark-data/processed/feature_pipeline


In [23]:
spark.stop()
print("SparkSession finalizada")


SparkSession finalizada
